# Explore the dialect choice-ablation parquets

Three variants are built by `build_data.py`:

| Variant | Prompt body | Choices column | Label column | Multi-correct |
|---|---|---|---|---|
| `explained_full` | extended (`GULF (e.g., Qatari)` etc.) | extended | extended | no |
| `explained_text_only` | extended | bare ADD (`Levant`, `GULF`...) | bare ADD | no |
| `arabench_choices` | ADD labels rewritten to AraBench (`Levant` -> `Lebanese`, `North Africa` -> `Tunisian, Moroccan`, ...) | 6 AraBench labels | canonical AraBench per coarse class (b1) | yes (via `accept_choices` + `metrics.target_indices`) |

Run this notebook from project root to eyeball one rendered sample per (variant, prompt), and to verify what lm-eval will receive as the gold for the `arabench_choices` variant before kicking off the eval.

Run order:
1. `uv run python ablations/dialect_explained_choices/build_data.py --variant all`
2. Run the cells below and confirm visually.
3. `sbatch ablations/dialect_explained_choices/slurm/run.sbatch`

In [1]:
import sys
from pathlib import Path

import pandas as pd

DATA_ROOT = Path('ablations/dialect_explained_choices/data')
YAML_ROOT = Path('ablations/dialect_explained_choices/eval_harness_tasks')
VARIANTS = ['explained_full', 'explained_text_only', 'arabench_choices']
PROMPT_IDS = [14102, 14783, 14784, 14790, 14851]

EXPECTED_BARE_ADD = {'Levant', 'North Africa', 'Egypt', 'GULF', 'MSA'}
EXPECTED_EXTENDED = {
    'Levant (e.g., Lebanese)', 'North Africa (e.g., Tunisian, Moroccan)',
    'Egypt', 'GULF (e.g., Qatari)', 'MSA',
}
EXPECTED_ARABENCH = {'MSA', 'Lebanese', 'Tunisian', 'Moroccan', 'Qatari', 'Egyptian'}

## Walk one sample per (variant, prompt)

For each prompt we print sample 0's rendered text, the full choices list, and the gold label. For `arabench_choices` we also print `accept_choices`.

In [2]:
for variant in VARIANTS:
    print('#' * 100)
    print(f'# VARIANT: {variant}')
    print('#' * 100)
    for pid in PROMPT_IDS:
        parquet = DATA_ROOT / variant / f'prompt_{pid}' / 'data.parquet'
        if not parquet.exists():
            print(f'[MISSING] {parquet}  -  run build_data.py --variant {variant}')
            continue
        df = pd.read_parquet(parquet)
        sample = df.iloc[0]
        print('=' * 100)
        print(f'{variant} / prompt_{pid}  (n_samples={len(df)})')
        print('=' * 100)
        print('--- rendered text (sample 0) ---')
        print(sample['text'])
        print()
        print('--- choices ---')
        for c in sample['choices']:
            print(f'  - {c}')
        print()
        print(f'--- gold label (sample 0): {sample["label"]!r} ---')
        if 'accept_choices' in df.columns:
            print(f'--- accept_choices (sample 0): {list(sample["accept_choices"])!r} ---')
        print()

####################################################################################################
# VARIANT: explained_full
####################################################################################################
explained_full / prompt_14102  (n_samples=9992)
--- rendered text (sample 0) ---
For this text: البطوله خلاص بين.. بايرن ميونخ + الأنتر.., I am not sure what dialect it is. Can you identify the dialect of the previous text given the following options: Levant (e.g., Lebanese), North Africa (e.g., Tunisian, Moroccan), Egypt, GULF (e.g., Qatari), MSA

--- choices ---
  - Levant (e.g., Lebanese)
  - North Africa (e.g., Tunisian, Moroccan)
  - Egypt
  - GULF (e.g., Qatari)
  - MSA

--- gold label (sample 0): 'Egypt' ---

explained_full / prompt_14783  (n_samples=9992)
--- rendered text (sample 0) ---
For the following Arabic text: البطوله خلاص بين.. بايرن ميونخ + الأنتر.., the most probable dialect (among Levant (e.g., Lebanese), North Africa (e.g., Tunisian, Morocca

## Verify `metrics.target_indices` for `arabench_choices`

`arabench_choices` uses `doc_to_target: !function metrics.target_indices` (see [eval_harness_tasks/arabench_choices/metrics.py](eval_harness_tasks/arabench_choices/metrics.py)). The function returns a list of indices into `choices` derived from each row's `accept_choices`; lm-eval auto-activates `multiple_target` mode from that list.

We import the function directly from the same file lm-eval will load and run it on real parquet rows (one per coarse class). For each coarse class we expect:

| ADD coarse | accept_choices | target_indices (into `['MSA', 'Lebanese', 'Tunisian', 'Moroccan', 'Qatari', 'Egyptian']`) |
|---|---|---|
| MSA | `['MSA']` | `[0]` |
| Levant | `['Lebanese']` | `[1]` |
| North Africa | `['Tunisian', 'Moroccan']` | `[2, 3]` |
| Egypt | `['Egyptian']` | `[5]` |
| GULF | `['Qatari']` | `[4]` |

In [3]:
# Import metrics.py from where lm-eval will load it.
metrics_dir = str(YAML_ROOT / 'arabench_choices')
if metrics_dir not in sys.path:
    sys.path.insert(0, metrics_dir)
if 'metrics' in sys.modules:
    del sys.modules['metrics']  # force re-import if we re-run
import metrics

EXPECTED_TARGET_INDICES = {
    # canonical AraBench label (the row's `label` column) -> expected indices
    'MSA':      [0],
    'Lebanese': [1],
    'Tunisian': [2, 3],  # North Africa: both Tunisian and Moroccan count
    'Qatari':   [4],
    'Egyptian': [5],
}

for pid in PROMPT_IDS:
    parquet = DATA_ROOT / 'arabench_choices' / f'prompt_{pid}' / 'data.parquet'
    if not parquet.exists():
        print(f'[SKIP] arabench_choices/prompt_{pid}: parquet missing')
        continue
    df = pd.read_parquet(parquet)
    print(f'--- arabench_choices / prompt_{pid} ---')
    for label, expected in EXPECTED_TARGET_INDICES.items():
        rows = df[df['label'] == label]
        if rows.empty:
            print(f'  [SKIP] no rows with label={label!r}')
            continue
        row = rows.iloc[0].to_dict()
        got = metrics.target_indices(row)
        match = '[OK]' if got == expected else '[FAIL]'
        print(
            f'  {match} label={label!r:10s} accept_choices={list(row["accept_choices"])!r:30s} '
            f'-> target_indices={got!r:10s} (expected {expected!r})'
        )
        assert got == expected, f'mismatch for {label} in prompt_{pid}'
print('\nAll target_indices outputs match expectation.')

--- arabench_choices / prompt_14102 ---
  [OK] label='MSA'      accept_choices=['MSA']                        -> target_indices=[0]        (expected [0])
  [OK] label='Lebanese' accept_choices=['Lebanese']                   -> target_indices=[1]        (expected [1])
  [OK] label='Tunisian' accept_choices=['Tunisian', 'Moroccan']       -> target_indices=[2, 3]     (expected [2, 3])
  [OK] label='Qatari'   accept_choices=['Qatari']                     -> target_indices=[4]        (expected [4])
  [OK] label='Egyptian' accept_choices=['Egyptian']                   -> target_indices=[5]        (expected [5])
--- arabench_choices / prompt_14783 ---
  [OK] label='MSA'      accept_choices=['MSA']                        -> target_indices=[0]        (expected [0])
  [OK] label='Lebanese' accept_choices=['Lebanese']                   -> target_indices=[1]        (expected [1])
  [OK] label='Tunisian' accept_choices=['Tunisian', 'Moroccan']       -> target_indices=[2, 3]     (expected [2, 3])
  

## Sanity checks

Assert that each variant's parquets match the spec: choice set is the right one, every label is in `choices`, and for `arabench_choices` every `accept_choices` entry is in `choices` and contains the row's `label`.

In [4]:
def check_variant(variant: str, expected_choice_set: set[str]) -> None:
    for pid in PROMPT_IDS:
        parquet = DATA_ROOT / variant / f'prompt_{pid}' / 'data.parquet'
        if not parquet.exists():
            print(f'  [SKIP] {variant}/prompt_{pid}: parquet missing')
            continue
        df = pd.read_parquet(parquet)
        # Same choice set every row, and it matches expected.
        seen = {tuple(c) for c in df['choices']}
        assert len(seen) == 1, f'{variant}/prompt_{pid}: choices vary across rows: {seen}'
        assert set(next(iter(seen))) == expected_choice_set, (
            f'{variant}/prompt_{pid}: choices {set(next(iter(seen)))} != expected {expected_choice_set}'
        )
        # Every label is in its row's choices.
        bad = df[~df.apply(lambda r: r['label'] in list(r['choices']), axis=1)]
        assert len(bad) == 0, f'{variant}/prompt_{pid}: {len(bad)} rows with label not in choices'
        # arabench_choices: every accept entry is in choices, and label is in accept.
        if 'accept_choices' in df.columns:
            bad_acc = df[~df.apply(
                lambda r: set(r['accept_choices']).issubset(set(r['choices']))
                          and r['label'] in list(r['accept_choices']),
                axis=1,
            )]
            assert len(bad_acc) == 0, f'{variant}/prompt_{pid}: {len(bad_acc)} rows with bad accept_choices'
        print(f'  [OK] {variant}/prompt_{pid}: {len(df)} rows')

print('--- explained_full ---')
check_variant('explained_full', EXPECTED_EXTENDED)
print('--- explained_text_only ---')
check_variant('explained_text_only', EXPECTED_BARE_ADD)
print('--- arabench_choices ---')
check_variant('arabench_choices', EXPECTED_ARABENCH)
print('\nAll checks passed.')

--- explained_full ---
  [OK] explained_full/prompt_14102: 9992 rows
  [OK] explained_full/prompt_14783: 9992 rows
  [OK] explained_full/prompt_14784: 9992 rows
  [OK] explained_full/prompt_14790: 9992 rows
  [OK] explained_full/prompt_14851: 9992 rows
--- explained_text_only ---
  [OK] explained_text_only/prompt_14102: 9992 rows
  [OK] explained_text_only/prompt_14783: 9992 rows
  [OK] explained_text_only/prompt_14784: 9992 rows
  [OK] explained_text_only/prompt_14790: 9992 rows
  [OK] explained_text_only/prompt_14851: 9992 rows
--- arabench_choices ---
  [OK] arabench_choices/prompt_14102: 9992 rows
  [OK] arabench_choices/prompt_14783: 9992 rows
  [OK] arabench_choices/prompt_14784: 9992 rows
  [OK] arabench_choices/prompt_14790: 9992 rows
  [OK] arabench_choices/prompt_14851: 9992 rows

All checks passed.


## Per-dialect sample-length comparison: AraBench vs ADD


In [5]:
import re
import statistics

ARABENCH_SRC = Path('experimental_hf_datasets/AraBench_dev')
ADD_SRC = Path('experimental_hf_datasets/Arabic_Dialects_Dataset')

# Either prompt works — the Arabic body is identical across prompts of a given dataset.
ARABENCH_PROMPT = 14561
ADD_PROMPT = 14102

# (display name, AraBench label(s), ADD coarse label). AraBench uses 'Morrocan' (sic).
CLASS_PAIRS = [
    ('MSA',          ['MSA'],                  'MSA'),
    ('Gulf',         ['Qatari'],               'GULF'),
    ('Levant',       ['Lebanese'],             'Levant'),
    ('North Africa', ['Tunisian', 'Morrocan'], 'North Africa'),
    ('Egyptian',     ['Egyptian'],             'Egypt'),
]

# Arabic + common punctuation/whitespace ranges, used to pull the dialectal body out
# of the rendered prompt (rest of the template is English).
ARABIC_RE = re.compile(r'[؀-ۿݐ-ݿࢠ-ࣿ\s،.؟]+')

def extract_arabic_body(rendered: str) -> str:
    runs = ARABIC_RE.findall(rendered)
    return max(runs, key=len).strip() if runs else ''

def stats(values: list[int]) -> dict:
    n = len(values)
    if n == 0:
        return {'n': 0, 'median': float('nan'), 'mean': float('nan'), 'p95': float('nan')}
    sv = sorted(values)
    return {
        'n': n,
        'median': statistics.median(sv),
        'mean': statistics.mean(sv),
        'p95': sv[int(0.95 * n)] if n > 1 else sv[0],
    }

ab = pd.read_parquet(ARABENCH_SRC / f'prompt_{ARABENCH_PROMPT}' / 'data.parquet').copy()
add = pd.read_parquet(ADD_SRC / f'prompt_{ADD_PROMPT}' / 'data.parquet').copy()

for df in (ab, add):
    df['_body'] = df['text'].apply(extract_arabic_body)
    df['_chars'] = df['_body'].str.len()
    df['_words'] = df['_body'].str.split().apply(len)

print('=' * 110)
print(f'PER-DIALECT TEXT-BODY LENGTH: AraBench_dev prompt_{ARABENCH_PROMPT}  vs  Arabic_Dialects_Dataset prompt_{ADD_PROMPT}')
print('=' * 110)
header = (
    f'{"Coarse class":<14s} {"Dataset":<9s} {"AraBench label(s)":<22s} {"n":>5s} '
    f'| {"chars med":>9s} {"mean":>5s} {"p95":>5s} '
    f'| {"words med":>9s} {"mean":>5s} {"p95":>5s}'
)
print(header)
print('-' * len(header))

for display, ab_labels, add_label in CLASS_PAIRS:
    ab_rows = ab[ab['label'].isin(ab_labels)]
    add_rows = add[add['label'] == add_label]

    abc = stats(ab_rows['_chars'].tolist())
    abw = stats(ab_rows['_words'].tolist())
    print(
        f'{display:<14s} {"AraBench":<9s} {",".join(ab_labels):<22s} {abc["n"]:>5d} '
        f'| {abc["median"]:>9.0f} {abc["mean"]:>5.0f} {abc["p95"]:>5.0f} '
        f'| {abw["median"]:>9.0f} {abw["mean"]:>5.0f} {abw["p95"]:>5.0f}'
    )
    addc = stats(add_rows['_chars'].tolist())
    addw = stats(add_rows['_words'].tolist())
    print(
        f'{display:<14s} {"ADD":<9s} {add_label:<22s} {addc["n"]:>5d} '
        f'| {addc["median"]:>9.0f} {addc["mean"]:>5.0f} {addc["p95"]:>5.0f} '
        f'| {addw["median"]:>9.0f} {addw["mean"]:>5.0f} {addw["p95"]:>5.0f}'
    )
    # Length-ratio per class (ADD / AraBench, on median chars).
    if abc['median'] and not pd.isna(abc['median']):
        ratio = addc['median'] / abc['median']
        print(f'{"":<14s} {"-> ratio":<9s} {"ADD med / AraBench med":<22s} {"":>5s} '
              f'| {ratio:>9.1f}x')
    print()

print('=' * 110)
print('AGGREGATE (excluding MSA): all-dialect combined')
print('=' * 110)
for name, df in [('AraBench dialects', ab[ab['label'] != 'MSA']),
                 ('ADD dialects',      add[add['label'] != 'MSA'])]:
    c = stats(df['_chars'].tolist())
    w = stats(df['_words'].tolist())
    print(
        f'  {name:<20s} n={c["n"]:>5d} | '
        f'chars med/mean/p95 = {c["median"]:>4.0f}/{c["mean"]:>4.0f}/{c["p95"]:>4.0f} | '
        f'words med/mean/p95 = {w["median"]:>4.0f}/{w["mean"]:>4.0f}/{w["p95"]:>4.0f}'
    )

print()
print('Key takeaway: ADD-MSA is ~6x longer than AraBench-MSA at the median, and ADD')
print('dialect bodies are ~3-5x longer than AraBench dialect bodies. This is a')
print('within-label register shift (phrasebook / short utterances vs broadcast news /')
print('long-form journalism), not a label-mapping problem.')

PER-DIALECT TEXT-BODY LENGTH: AraBench_dev prompt_14561  vs  Arabic_Dialects_Dataset prompt_14102
Coarse class   Dataset   AraBench label(s)          n | chars med  mean   p95 | words med  mean   p95
-----------------------------------------------------------------------------------------------------
MSA            AraBench  MSA                     1597 |        34    40    85 |         7     9    17
MSA            ADD       MSA                      999 |       197   264   718 |        35    46   124
               -> ratio  ADD med / AraBench med       |       5.8x

Gulf           AraBench  Qatari                  2901 |        25    30    68 |         5     6    13
Gulf           ADD       GULF                    1672 |        98   176   621 |        18    33   115
               -> ratio  ADD med / AraBench med       |       3.9x

Levant         AraBench  Lebanese                1302 |        25    29    59 |         5     6    12
Levant         ADD       Levant                  175

## Eyeball random samples: AraBench vs ADD, per dialect

Pull a reproducible random subset from each dataset for direct qualitative inspection. Set `DIALECT` to one of `MSA`, `Gulf`, `Levant`, `North Africa`, `Egyptian` (default `MSA`); the cell pairs the matching AraBench label(s) with the corresponding ADD coarse label (AraBench `Tunisian`+`Morrocan` -> ADD `North Africa`, AraBench `Qatari` -> ADD `GULF`, etc.). Also tunable: `SEED`, `N_SAMPLES`.

Requires the previous cell to have run (uses the `ab` / `add` DataFrames with `_body` / `_chars` / `_words` columns).

In [9]:
import random
from html import escape
from IPython.display import HTML, display

# --- Tunables ---------------------------------------------------------------
SEED = 40
N_SAMPLES = 5
DIALECT = 'MSA'  # one of: 'MSA', 'Gulf', 'Levant', 'North Africa', 'Egyptian'
# ----------------------------------------------------------------------------

# AraBench uses 'Morrocan' (sic). North Africa pools Tunisian + Morrocan.
DIALECT_TO_LABELS = {
    # coarse name : (AraBench labels, ADD label)
    'MSA':          (['MSA'],                  'MSA'),
    'Gulf':         (['Qatari'],               'GULF'),
    'Levant':       (['Lebanese'],             'Levant'),
    'North Africa': (['Tunisian', 'Morrocan'], 'North Africa'),
    'Egyptian':     (['Egyptian'],             'Egypt'),
}
if DIALECT not in DIALECT_TO_LABELS:
    raise ValueError(f'Unknown DIALECT={DIALECT!r}. Choose from {list(DIALECT_TO_LABELS)}.')
ab_labels, add_label = DIALECT_TO_LABELS[DIALECT]

rng = random.Random(SEED)

ab_pool  = ab[ab['label'].isin(ab_labels)].reset_index(drop=True)
add_pool = add[add['label'] == add_label].reset_index(drop=True)

ab_idx  = rng.sample(range(len(ab_pool)),  min(N_SAMPLES, len(ab_pool)))
add_idx = rng.sample(range(len(add_pool)), min(N_SAMPLES, len(add_pool)))

def _cell(row) -> str:
    if row is None:
        return '<td style="border:1px solid #ccc; padding:8px;"></td>'
    meta = (f'<span style="color:#666; font-size:0.85em;">'
            f'chars={row["_chars"]} · words={row["_words"]} · label={row["label"]}</span>')
    return (
        '<td style="border:1px solid #ccc; padding:8px; vertical-align:top; '
        'max-width:480px; word-wrap:break-word;">'
        f'<div dir="rtl" lang="ar" style="text-align:right; '
        'font-family:\'Segoe UI\',\'Tahoma\',sans-serif; line-height:1.6;">'
        f'{escape(row["_body"])}</div>'
        f'<div style="margin-top:6px;">{meta}</div>'
        '</td>'
    )

rows_html = []
for i in range(N_SAMPLES):
    ab_row  = ab_pool.iloc[ab_idx[i]]   if i < len(ab_idx)  else None
    add_row = add_pool.iloc[add_idx[i]] if i < len(add_idx) else None
    rows_html.append(
        '<tr>'
        f'<td style="border:1px solid #ccc; padding:4px 8px; text-align:center; '
        f'font-weight:bold; vertical-align:top;">{i + 1}</td>'
        f'{_cell(ab_row)}{_cell(add_row)}'
        '</tr>'
    )

header = (
    '<thead>'
    '<tr style="background:#eef;">'
    '<th style="border:1px solid #ccc; padding:6px 8px; width:30px;">#</th>'
    f'<th style="border:1px solid #ccc; padding:6px 8px;">AraBench &nbsp;'
    f'<span style="color:#666; font-weight:normal; font-size:0.85em;">'
    f'(label={"+".join(ab_labels)}, pool n={len(ab_pool)})</span></th>'
    f'<th style="border:1px solid #ccc; padding:6px 8px;">ADD &nbsp;'
    f'<span style="color:#666; font-weight:normal; font-size:0.85em;">'
    f'(label={add_label}, pool n={len(add_pool)})</span></th>'
    '</tr></thead>'
)
caption = (
    f'<p style="margin:6px 0; font-size:0.9em; color:#444;">'
    f'<b>{N_SAMPLES} random samples per dataset — DIALECT={DIALECT}</b> &nbsp; (seed={SEED})'
    '</p>'
)
table = (
    caption
    + '<table style="border-collapse:collapse; table-layout:fixed; width:100%;">'
    + header
    + '<tbody>' + ''.join(rows_html) + '</tbody>'
    + '</table>'
)
display(HTML(table))

#,"AraBench (label=MSA, pool n=1597)","ADD (label=MSA, pool n=999)"
1,لا تزعج .chars=9 · words=3 · label=MSA,طالما أن البرلمان مصرة على إسقاط الرئيس وبطريقة غير عادلة فإن الشعب من الذين يسمون أنفسهم أنصاري سوف يأتون إلى جاكرتاchars=117 · words=21 · label=MSA
2,ما هو الوقت الذي قلت أننا ينبغي أن نسلم ورقتنا بحلوله ؟chars=55 · words=12 · label=MSA,نجح محمد علي في تنفيذ خططه الطموحة في مجال التصنيع مكنته من الاستمرار في حروبه المتعد دة وتحديه للإمبراطورية العثمانية وهو ما نبه الغربة لخطورة مشروعه الطموح على المصالح الغربية فكان القرار بضرورة وقف هذا النجاح بأي ثمن وكانت معاهدة عام ألف وثمانمائة وأربعين هي الحد الفاصل ما بين ذروة القمة التي وصلتها الصناعة المصرية آنذاك وبدايات الأفولchars=340 · words=58 · label=MSA
3,ففتح فاه وعلمهم قائلاchars=21 · words=4 · label=MSA,دولارا للبرميل أكد مسؤول نفطي يمني أن اليمن رفع سعر وقود الديزل اعتبارا من اليوم بنسبة سبعين في المائة في إطار برنامج الإصلاح الاقتصادي الذي ينفذه منذ عام خمسة وتسعين وأوضح المسؤول أن دعم أسعار الديزل كلفت البدايات ثلاثة مليارات ونصف المليار ريال شهريا الأمر الذي أدى إلى انتشار عمليات تهريبه إلى الدول المجاورة والتزام الحكومة اليمنية تدعم أنواع أخرى من الوقود مثل الكيروزان والمازوت المستخدم في توليد الكهرباء من جهة أخرى قال مصدر حكومي إن الحكومة ستتخذ قرارا بزيادة المرتبات للحد من تأثير رفع سعر الديزل على الموظفينchars=519 · words=89 · label=MSA
4,أريد مشبك شعر .chars=15 · words=4 · label=MSA,صحيفة هآرتس الإسرائيلية تعتبر أن كثافة عمليات القتل المتبادلة تعميق الكراهية والرغبة في الانتقام في وقت يحتاج الموقف فيه إلى ضبط المشاعر والأعصاب خاصة أن الحكومة الإسرائيلية لا تملك حاليا أي مشروع سياسي أضافت الصحيفة أن هذا الوضع يتطلب من حزب العمل المشارك في حكومة شارون أن يعيد النظر في استمرار مشاركته في هذه الحكومة التي لا تملك إلا الخيار العسكريchars=351 · words=61 · label=MSA
5,لقد طلبت رقماً خطأ .chars=20 · words=5 · label=MSA,وتتعرض إيران على البلدان الغربية إمكانية تزويدها بآليات التكنولوجيا النووية المدنية في مقابل توقيع طهران على اتفاقية حظر التجارب النووية إلا أن البلدان الغربية في مقدمة بلدان الاتحاد ترفض الطلب الإيراني واللافت هنا هو أن الخلافات القائمة منذ أعوام لم تعد تمثل وحاجزا أمام تطبيع العلاقات بين الجانبين وحرصهما على إبرام اتفاقية للتعاون الاقتصاديchars=343 · words=54 · label=MSA
